# Notebook 30. IMERG precipitation versus catalogued JPCZ convergence

This notebook uses the **already calculated, saved JPCZ convergence signal in the merged catalog**. It does not reopen ERA5 or recalculate divergence. Its only remote-data task is to retrieve GPM IMERG Final V07 precipitation for the event windows and save compact, analysis-ready regional precipitation data to Google Drive.

## Established predictor

The catalog field `event_peak_D_1e5_s-1` is the area-weighted, trailing 12-hour 925-hPa divergence for the digitized Shinoda JPCZ polygon at each event peak. Notebook 30 uses its negative, `-D12`, as convergence strength: larger positive values mean stronger convergence. This is the same signal that detected the events, not a new calculation.

The four figures use that one established predictor against two IMERG outcomes (accumulation and mean rate) in two precipitation regions (the JPCZ polygon and the coastal wedge). A separate coastal-divergence predictor is deliberately out of scope here; it would require a separate reproducible analysis rather than silently mixing metrics.

## Workflow

1. Authenticate once with NASA Earthdata.
2. Create and save a Drive event-data plan from the expanded merged catalog.
3. Collect and checkpoint IMERG one event (or a small batch) at a time.
4. Verify that the Drive inventory is complete.
5. Run the final statistics and presentation plots from the saved tables.

IMERG Final V07 begins in June 2000, so events whose full precipitation window begins before then are excluded. Raw NASA HDF files are not duplicated in Drive; NASA retains the authoritative archive. Drive stores the complete, compact regional half-hourly series, the one-row-per-event precipitation metrics, and the request inventory needed for this analysis.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git'
BRANCH = 'codex/notebook16-pcolormesh'
REPO_DIR = '/content/JPCZcatalog'
FORCE_REFRESH_REPO = True
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if FORCE_REFRESH_REPO and Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')
print('Repository branch:', BRANCH)
print('Drive checkpoint root:', DRIVE_ROOT)

In [ ]:
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from jpcz_catalog.config import BoundingBox, JPCZ_POLYGON_VERTICES
from jpcz_catalog.imerg import parse_imerg_granule_start, read_precipitation_cal_subset, region_mean_rates

# ----- User controls -----
CATALOG_OVERRIDE_PATH = None  # Leave None to use the current expanded Drive catalog.
EVENTS_PER_COLLECTION_RUN = 1  # Start with one event; use None only after a successful test.
ALLOW_PARTIAL_ANALYSIS = False  # Keep False: final statistics unlock only when all eligible events are saved.
DELETE_LOCAL_GRANULES_AFTER_CHECKPOINT = True

IMERG_FIRST_VALID_TIME = pd.Timestamp('2000-06-01 00:00:00')
MIN_TEMPORAL_COVERAGE = 0.90

COASTAL_WEDGE_VERTICES = (
    (133.05, 35.55), (136.05, 35.55), (139.55, 39.00), (139.55, 42.55),
)
REGIONS = {'jpcz_polygon': JPCZ_POLYGON_VERTICES, 'coastal_wedge': COASTAL_WEDGE_VERTICES}
IMERG_READ_DOMAIN = BoundingBox(lon_min=128.0, lon_max=141.0, lat_min=35.0, lat_max=43.0)

DRIVE_ANALYSIS_DIR = DRIVE_ROOT / 'imerg_precipitation_convergence'
DRIVE_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
IMERG_RATE_PATH = DRIVE_ANALYSIS_DIR / 'imerg_regional_halfhourly_rates.csv'
IMERG_EVENT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_regional_precipitation.csv'
EVENT_METRICS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_precipitation_convergence_metrics.csv'
STATS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_statistics.csv'
PLOT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_scatter.png'
MANIFEST_PATH = DRIVE_ANALYSIS_DIR / 'imerg_catalog_run_manifest.csv'
EVENT_PLAN_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_collection_plan.csv'
LOCAL_GRANULE_DIR = Path('/content/imerg_event_granules')

def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

def read_checkpoint(path, parse_dates=()):
    path = Path(path)
    return pd.read_csv(path, parse_dates=list(parse_dates)) if path.exists() else pd.DataFrame()

def merge_checkpoint(existing, fresh, key):
    combined = fresh.copy() if existing.empty else pd.concat([existing, fresh], ignore_index=True)
    return combined.drop_duplicates(key, keep='last').sort_values(key).reset_index(drop=True)

def write_event_plan(events, event_metrics):
    completed = set(event_metrics.loc[event_metrics['status'].eq('ok'), 'event_id'].astype(str)) if {'event_id', 'status'}.issubset(event_metrics.columns) else set()
    plan = events[['event_id', 'event_start', 'event_end', 'event_peak', 'precip_window_start', 'precip_window_end_exclusive', 'precip_window_hours', 'jpcz_polygon_convergence_1e5_s-1']].copy()
    plan['imerg_product'] = 'GPM_3IMERGHH V07 Final / Grid/precipitationCal'
    plan['collection_status'] = np.where(plan['event_id'].isin(completed), 'complete', 'pending')
    atomic_csv(plan, EVENT_PLAN_PATH)
    return plan

def catalog_candidates():
    return [
        Path(CATALOG_OVERRIDE_PATH) if CATALOG_OVERRIDE_PATH else None,
        DRIVE_ROOT / 'jpcz_catalog_ndjf_merged_12h.csv',
        Path('outputs/verification/jpcz_catalog_ndjf_merged_12h.csv'),
    ]

In [ ]:
catalog_path = next((path for path in catalog_candidates() if path is not None and path.exists()), None)
if catalog_path is None:
    raise FileNotFoundError('No merged catalog found. Run Notebook 06 first so the merged CSV is in Google Drive.')

catalog = pd.read_csv(catalog_path, parse_dates=['event_start', 'event_end', 'event_peak']).sort_values('event_start').reset_index(drop=True)
if 'event_peak_D_1e5_s-1' in catalog.columns:
    catalog['jpcz_polygon_convergence_1e5_s-1'] = -pd.to_numeric(catalog['event_peak_D_1e5_s-1'], errors='coerce')
elif 'event_peak_D_s-1' in catalog.columns:
    catalog['jpcz_polygon_convergence_1e5_s-1'] = -pd.to_numeric(catalog['event_peak_D_s-1'], errors='coerce') * 1e5
else:
    raise KeyError('The merged catalog needs event_peak_D_1e5_s-1 or event_peak_D_s-1. Rerun Notebook 06 from the current Notebook 04 catalog.')

catalog['event_id'] = catalog['event_peak'].dt.strftime('%Y%m%dT%H%M')
if catalog['event_id'].duplicated().any():
    raise ValueError('Event peaks must be unique before the IMERG event workflow can run.')
catalog['precip_window_start'] = catalog['event_start'] - pd.Timedelta(hours=11)
catalog['precip_window_end_exclusive'] = catalog['event_end'] + pd.Timedelta(hours=1)
catalog['precip_window_hours'] = (catalog['precip_window_end_exclusive'] - catalog['precip_window_start']).dt.total_seconds() / 3600
catalog_for_imerg = catalog.loc[catalog['precip_window_start'] >= IMERG_FIRST_VALID_TIME].copy()

manifest = pd.DataFrame([{
    'catalog_source': str(catalog_path), 'merged_catalog_rows': len(catalog),
    'imerg_eligible_events': len(catalog_for_imerg), 'excluded_before_imerg': len(catalog) - len(catalog_for_imerg),
    'first_peak_utc': catalog['event_peak'].min(), 'last_peak_utc': catalog['event_peak'].max(),
}])
atomic_csv(manifest, MANIFEST_PATH)
existing_imerg_events = read_checkpoint(IMERG_EVENT_PATH, parse_dates=['event_peak'])
event_plan = write_event_plan(catalog_for_imerg, existing_imerg_events)

print('Catalog selected:', catalog_path)
print(f'Merged catalog: {len(catalog)} events; IMERG-eligible: {len(catalog_for_imerg)}; excluded before June 2000: {len(catalog) - len(catalog_for_imerg)}')
print('Predictor: saved catalogued JPCZ-polygon -D12 convergence, not recalculated ERA5.')
print('Peak-date coverage:', catalog['event_peak'].min(), 'to', catalog['event_peak'].max())
print('Saved event collection plan:', EVENT_PLAN_PATH)
display(catalog_for_imerg[['event_id', 'event_start', 'event_end', 'event_peak', 'duration_hours', 'jpcz_polygon_convergence_1e5_s-1']].head())

## Phase A — NASA Earthdata authentication

Run this cell once per Colab session before collecting data. It is deliberately separate from the download cell, so your login is handled first and no credentials are written into the notebook or Google Drive.

In [ ]:
import earthaccess
earthaccess.login()
earthdata_authenticated = True
print('NASA Earthdata authentication completed for this Colab session.')

## Phase B — Retrieve and checkpoint IMERG event data

After successful authentication, run this cell. It will request only the next pending event's half-hourly IMERG files, reduce every field immediately to the two regional means, save the compact regional-rate table, and save the completed event row. Rerun it to advance. Set `EVENTS_PER_COLLECTION_RUN = None` only if you want it to work through all remaining events in one long, resumable run.

In [ ]:
def precipitation_metric_row(event, rates):
    expected = pd.date_range(event.precip_window_start, event.precip_window_end_exclusive, freq='30min', inclusive='left')
    indexed = rates.set_index('time').sort_index() if 'time' in rates.columns else pd.DataFrame(index=pd.DatetimeIndex([]))
    window = indexed.reindex(expected)
    row = {'event_id': event.event_id, 'event_peak': event.event_peak, 'imerg_expected_halfhours': len(expected), 'imerg_window_hours': len(expected) * 0.5}
    all_complete = True
    for name in REGIONS:
        column = f'{name}_rate_mm_hr'
        valid = int(window[column].notna().sum()) if column in window else 0
        coverage = valid / len(expected)
        accumulation = window[column].sum(skipna=True) * 0.5 if coverage >= MIN_TEMPORAL_COVERAGE and column in window else np.nan
        row[f'{name}_imerg_valid_halfhours'] = valid
        row[f'{name}_imerg_coverage_fraction'] = coverage
        row[f'{name}_imerg_accumulation_mm'] = accumulation
        row[f'{name}_imerg_mean_rate_mm_hr'] = accumulation / (len(expected) * 0.5) if pd.notna(accumulation) else np.nan
        all_complete = all_complete and pd.notna(accumulation)
    row['status'] = 'ok' if all_complete else 'incomplete'
    return row

imerg_events = read_checkpoint(IMERG_EVENT_PATH, parse_dates=['event_peak'])
completed_ids = set(imerg_events.loc[imerg_events['status'].eq('ok'), 'event_id'].astype(str)) if {'event_id', 'status'}.issubset(imerg_events.columns) else set()
pending = catalog_for_imerg.loc[~catalog_for_imerg['event_id'].isin(completed_ids)].copy()
imerg_rates = read_checkpoint(IMERG_RATE_PATH, parse_dates=['time'])
print(f'IMERG checkpoint: {len(completed_ids)} complete events; {len(pending)} remaining. Regional-rate cache: {len(imerg_rates)} half-hours.')

if not globals().get('earthdata_authenticated', False):
    raise RuntimeError('Run the Phase A Earthdata authentication cell before collecting IMERG.')
if not pending.empty:
    batch = pending if EVENTS_PER_COLLECTION_RUN is None else pending.head(int(EVENTS_PER_COLLECTION_RUN))
    for position, event in enumerate(batch.itertuples(index=False), start=1):
        expected = pd.date_range(event.precip_window_start, event.precip_window_end_exclusive, freq='30min', inclusive='left')
        existing_times = pd.DatetimeIndex(imerg_rates['time']) if not imerg_rates.empty else pd.DatetimeIndex([])
        wanted = set(expected.difference(existing_times))
        print(f'IMERG {position}/{len(batch)}: {event.event_id} | {len(wanted)}/{len(expected)} half-hours missing')
        if wanted:
            local_event_dir = LOCAL_GRANULE_DIR / event.event_id
            local_event_dir.mkdir(parents=True, exist_ok=True)
            results = earthaccess.search_data(
                short_name='GPM_3IMERGHH', version='07',
                temporal=(pd.Timestamp(event.precip_window_start).isoformat(), pd.Timestamp(event.precip_window_end_exclusive).isoformat()),
                count=500,
            )
            files = earthaccess.download(results, local_path=local_event_dir, threads=2)
            fresh_rows = []
            for file_path in files:
                try:
                    timestamp = pd.Timestamp(parse_imerg_granule_start(file_path))
                    if timestamp in wanted:
                        rate_field = read_precipitation_cal_subset(file_path, domain=IMERG_READ_DOMAIN)
                        fresh_rows.append({'time': timestamp, **{f'{name}_rate_mm_hr': value for name, value in region_mean_rates(rate_field, REGIONS).items()}})
                except Exception as error:
                    print(f'  skipped {Path(file_path).name}: {type(error).__name__}: {error}')
            if fresh_rows:
                imerg_rates = merge_checkpoint(imerg_rates, pd.DataFrame(fresh_rows), 'time')
                atomic_csv(imerg_rates, IMERG_RATE_PATH)
                print(f'  saved {len(fresh_rows)} half-hours; cache now has {len(imerg_rates)} rows')
            if DELETE_LOCAL_GRANULES_AFTER_CHECKPOINT:
                shutil.rmtree(local_event_dir, ignore_errors=True)
        metric = pd.DataFrame([precipitation_metric_row(event, imerg_rates)])
        if metric.loc[0, 'status'] == 'ok':
            imerg_events = merge_checkpoint(imerg_events, metric, 'event_id')
            atomic_csv(imerg_events, IMERG_EVENT_PATH)
            print(f'  saved completed precipitation event: {event.event_id}')
        else:
            print('  event remains incomplete and will be retried on the next run.')
        gc.collect()

imerg_events = read_checkpoint(IMERG_EVENT_PATH, parse_dates=['event_peak'])
event_plan = write_event_plan(catalog_for_imerg, imerg_events)
print('Drive collection plan updated:', EVENT_PLAN_PATH)
display(event_plan['collection_status'].value_counts().rename_axis('status').reset_index(name='event_count'))
display(imerg_events.tail())

## Phase C — Verify the backed-up event inventory

This cell reads only Drive files. With the default `ALLOW_PARTIAL_ANALYSIS = False`, the final statistics and figures remain locked until every IMERG-eligible event in the saved collection plan is complete.

In [ ]:
imerg_events = read_checkpoint(IMERG_EVENT_PATH, parse_dates=['event_peak'])
if not {'event_id', 'event_peak'}.issubset(imerg_events.columns):
    imerg_events = pd.DataFrame(columns=['event_id', 'event_peak'])
analysis = catalog_for_imerg.merge(imerg_events, on=['event_id', 'event_peak'], how='left')
atomic_csv(analysis, EVENT_METRICS_PATH)
required = ['jpcz_polygon_convergence_1e5_s-1', 'jpcz_polygon_imerg_mean_rate_mm_hr', 'coastal_wedge_imerg_mean_rate_mm_hr']
complete = int(analysis.dropna(subset=required).shape[0]) if all(column in analysis.columns for column in required) else 0
all_data_ready = complete == len(analysis) and len(analysis) > 0
event_plan = write_event_plan(catalog_for_imerg, imerg_events)
print(f'Joined event table: {complete}/{len(analysis)} events have catalogued convergence and both regional IMERG rate metrics.')
print('Final-analysis readiness:', 'READY' if all_data_ready else 'NOT READY — continue Phase B collection.')
display(event_plan['collection_status'].value_counts().rename_axis('status').reset_index(name='event_count'))
display(analysis.head())

In [ ]:
def association_statistics(frame, x_column, y_column, region, measure):
    if x_column not in frame or y_column not in frame:
        return {'region': region, 'precipitation_measure': measure, 'n': 0, 'status': 'checkpoint data not built yet'}
    sample = frame[[x_column, y_column]].dropna()
    n = len(sample)
    if n < 4:
        return {'region': region, 'precipitation_measure': measure, 'n': n, 'status': 'need at least four complete events'}
    result = stats.pearsonr(sample[x_column], sample[y_column])
    fit = stats.linregress(sample[x_column], sample[y_column])
    z = np.arctanh(result.statistic)
    r_margin = stats.norm.ppf(0.975) / np.sqrt(n - 3)
    r_low, r_high = np.tanh([z - r_margin, z + r_margin])
    slope_margin = stats.t.ppf(0.975, n - 2) * fit.stderr
    return {
        'region': region, 'precipitation_measure': measure, 'n': n, 'status': 'ok',
        'pearson_r': result.statistic, 'r_95ci_low': r_low, 'r_95ci_high': r_high, 'r_two_sided_p': result.pvalue,
        'slope': fit.slope, 'slope_95ci_low': fit.slope - slope_margin, 'slope_95ci_high': fit.slope + slope_margin,
        'slope_two_sided_p': fit.pvalue, 'intercept': fit.intercept, 'r_squared': fit.rvalue ** 2,
        'null_decision_alpha_0.05': 'reject H0' if result.pvalue < 0.05 else 'fail to reject H0',
    }

specifications = []
for region, label in [('jpcz_polygon', 'JPCZ polygon'), ('coastal_wedge', 'Coastal wedge')]:
    specifications.extend([
        (label, 'IMERG accumulation (mm)', 'jpcz_polygon_convergence_1e5_s-1', f'{region}_imerg_accumulation_mm'),
        (label, 'IMERG mean rate (mm h-1)', 'jpcz_polygon_convergence_1e5_s-1', f'{region}_imerg_mean_rate_mm_hr'),
    ])
if not all_data_ready and not ALLOW_PARTIAL_ANALYSIS:
    statistics_table = pd.DataFrame([
        {'region': region, 'precipitation_measure': measure, 'n': 0, 'status': 'waiting for complete Drive event inventory'}
        for region, measure, _, _ in specifications
    ])
    print('Final statistics are locked until the IMERG collection plan is complete.')
else:
    statistics_table = pd.DataFrame([association_statistics(analysis, x, y, region, measure) for region, measure, x, y in specifications])
atomic_csv(statistics_table, STATS_PATH)
display(statistics_table.round(4))

In [ ]:
def plot_association(ax, x_column, y_column, title, ylabel, summary):
    if summary.get('status') != 'ok':
        ax.text(0.5, 0.5, 'Waiting for the complete IMERG event inventory', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return None
    sample = analysis[[x_column, y_column, 'duration_hours']].dropna()
    points = ax.scatter(sample[x_column], sample[y_column], c=sample['duration_hours'], cmap='viridis', s=38, edgecolor='white', linewidth=0.35)
    fit = stats.linregress(sample[x_column], sample[y_column])
    xline = np.linspace(sample[x_column].min(), sample[x_column].max(), 100)
    ax.plot(xline, fit.intercept + fit.slope * xline, color='#c0392b', linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Catalogued 925-hPa convergence, -D12 (10^-5 s^-1)')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.text(0.03, 0.97, f"n={int(summary['n'])}\nr={summary['pearson_r']:.2f} ({summary['r_95ci_low']:.2f}, {summary['r_95ci_high']:.2f})\np={summary['r_two_sided_p']:.3g}; {summary['null_decision_alpha_0.05']}", va='top', transform=ax.transAxes, fontsize=9, bbox={'facecolor': 'white', 'alpha': 0.9})
    return points

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
last_points = None
for ax, (region, measure, x_column, y_column) in zip(axes.flat, specifications):
    summary = statistics_table.loc[(statistics_table['region'] == region) & (statistics_table['precipitation_measure'] == measure)].iloc[0].to_dict()
    result = plot_association(ax, x_column, y_column, f'{region}: {measure}', measure, summary)
    if result is not None:
        last_points = result
if last_points is not None:
    fig.colorbar(last_points, ax=axes, shrink=0.82, pad=0.02, label='Merged event duration (h)')
fig.suptitle('IMERG Final V07 precipitation versus catalogued JPCZ convergence', fontsize=15)
fig.savefig(PLOT_PATH, dpi=220, bbox_inches='tight')
plt.show()
print('Figure saved:', PLOT_PATH)

## Methods wording for the presentation

For each merged JPCZ episode, we used the detector's saved 12-hour trailing, area-weighted 925-hPa divergence at the event peak and multiplied it by -1 so larger values indicate stronger JPCZ-polygon convergence. We obtained GPM IMERG Final V07 gauge-calibrated precipitation (`precipitationCal`; half-hourly 0.1 degree grid), calculated cosine-latitude-area-weighted precipitation rates over the digitized JPCZ polygon and coastal wedge, and calculated event accumulation by summing rate times 0.5 hour and event mean rate by dividing by the event-window duration. We evaluated the precipitation-convergence association using two-sided Pearson correlation and ordinary least-squares regression, reporting r, 95 percent confidence intervals, slope, R squared, and p values at alpha = 0.05.